# Azure OpenAI Response API — Stateless Cross-Instance Test

This notebook tests whether the **Response API state** (i.e. `response_id`) is portable across
different Azure OpenAI (Foundry) instances deployed in **Sweden Central**.

## Test Plan

| # | Test | Description |
|---|------|-------------|
| 1 | **Baseline** | Send a request to Instance 1, retrieve the response using the same instance. |
| 2 | **Cross-Instance Retrieve** | Send a request to Instance 1, retrieve the response from Instance 2 & 3. |
| 3 | **Cross-Instance Chaining** | Send a request to Instance 1, chain with `previous_response_id` on Instance 2 & 3. |
| 4 | **Background Mode Baseline** | Send a `background=True` request to Instance 1, poll until complete, retrieve from same instance. |
| 5 | **Background Cross-Instance Retrieve** | Send `background=True` to Instance 1, retrieve from Instance 2 & 3. |
| 6 | **Background Cross-Instance Chaining** | Send `background=True` to Instance 1, chain on Instance 2 & 3. |

Each test logs the outcome (success / failure + error details).

## 0. Setup & Configuration

In [1]:
# Install / upgrade dependencies
%pip install --upgrade openai python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import time
from datetime import datetime
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Validate that all required env vars are set
required_vars = [
    "AZURE_OPENAI_ENDPOINT_1", "AZURE_OPENAI_API_KEY_1",
    "AZURE_OPENAI_ENDPOINT_2", "AZURE_OPENAI_API_KEY_2",
    "AZURE_OPENAI_ENDPOINT_3", "AZURE_OPENAI_API_KEY_3",
    "AZURE_OPENAI_DEPLOYMENT_NAME",
]
missing = [v for v in required_vars if not os.getenv(v)]
if missing:
    raise EnvironmentError(f"Missing environment variables: {', '.join(missing)}")

DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

print(f"Deployment name : {DEPLOYMENT}")
print(f"Instance 1      : {os.getenv('AZURE_OPENAI_ENDPOINT_1')}")
print(f"Instance 2      : {os.getenv('AZURE_OPENAI_ENDPOINT_2')}")
print(f"Instance 3      : {os.getenv('AZURE_OPENAI_ENDPOINT_3')}")

Deployment name : gpt-4-1-mini
Instance 1      : https://ais-fnd1-6ohufxbigpdyo.openai.azure.com/
Instance 2      : https://ais-fnd2-6ohufxbigpdyo.openai.azure.com/
Instance 3      : https://ais-fnd3-6ohufxbigpdyo.openai.azure.com/


In [3]:
def make_client(instance_num: int) -> OpenAI:
    """Create an OpenAI client for the given Foundry instance (1, 2, or 3)."""
    endpoint = os.getenv(f"AZURE_OPENAI_ENDPOINT_{instance_num}").rstrip("/")
    api_key = os.getenv(f"AZURE_OPENAI_API_KEY_{instance_num}")
    return OpenAI(
        api_key=api_key,
        base_url=f"{endpoint}/openai/v1/",
        default_headers={"api-key": api_key},
    )

client1 = make_client(1)
client2 = make_client(2)
client3 = make_client(3)

clients = {1: client1, 2: client2, 3: client3}

print("All three OpenAI clients created successfully.")

All three OpenAI clients created successfully.


In [4]:
# ── Helper utilities ──────────────────────────────────────────────────

results = []  # collects test outcomes


def print_response_json(response, label: str = "Response"):
    """Pretty-print a Response API object as formatted JSON."""
    print(f"\n📋 {label} (JSON):")
    print(json.dumps(response.model_dump(), indent=2, default=str))
    print()


def log_result(test_name: str, success: bool, details: str = ""):
    """Append a test result and print it."""
    status = "PASS" if success else "FAIL"
    entry = {
        "test": test_name,
        "status": status,
        "details": details,
        "timestamp": datetime.utcnow().isoformat(),
    }
    results.append(entry)
    colour = "\033[92m" if success else "\033[91m"
    print(f"{colour}[{status}]\033[0m {test_name}")
    if details:
        print(f"       ↳ {details}")


def wait_for_background(client: OpenAI, response_id: str, timeout: int = 120) -> object:
    """Poll a background response until it reaches a terminal state."""
    start = time.time()
    resp = client.responses.retrieve(response_id)
    while resp.status in ("queued", "in_progress"):
        if time.time() - start > timeout:
            raise TimeoutError(f"Background response {response_id} did not complete within {timeout}s")
        time.sleep(2)
        resp = client.responses.retrieve(response_id)
    return resp

---
## 1. Baseline — Create & Retrieve on Same Instance

In [5]:
# Test 1: Create a response on Instance 1 and retrieve it from the same instance
print("Creating response on Instance 1...")
response1 = client1.responses.create(
    model=DEPLOYMENT,
    input="What is the capital of France? Answer in one word.",
)

response1_id = response1.id
print(f"Response ID  : {response1_id}")
print(f"Status       : {response1.status}")
print(f"Output       : {response1.output_text}")
print_response_json(response1, "Create Response (Instance 1)")

# Retrieve the same response from Instance 1
print("Retrieving response from Instance 1 (same instance)...")
try:
    retrieved = client1.responses.retrieve(response1_id)
    print_response_json(retrieved, "Retrieved Response (Instance 1)")
    log_result(
        "1. Baseline — create & retrieve on Instance 1",
        success=True,
        details=f"Retrieved output: {retrieved.output_text}",
    )
except Exception as e:
    log_result("1. Baseline — create & retrieve on Instance 1", success=False, details=str(e))

Creating response on Instance 1...
Response ID  : resp_0a1c9e7331341b450069cba80095588194bcc1c817f57699f5
Status       : completed
Output       : Paris

📋 Create Response (Instance 1) (JSON):
{
  "id": "resp_0a1c9e7331341b450069cba80095588194bcc1c817f57699f5",
  "created_at": 1774954496.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4-1-mini",
  "object": "response",
  "output": [
    {
      "id": "msg_0a1c9e7331341b450069cba800e0b08194b00b430bfc572f35",
      "content": [
        {
          "annotations": [],
          "text": "Paris",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": null
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 1.0,
  "background": false,
  "completed_at": 1774954496.0,
  "conversation": null,
  "max_output_tok

---
## 2. Cross-Instance Retrieve — Retrieve Response from Instance 2 & 3

In [6]:
# Test 2a: Try to retrieve the response created on Instance 1 from Instance 2
print(f"Attempting to retrieve response {response1_id} from Instance 2...")
try:
    retrieved_2 = client2.responses.retrieve(response1_id)
    print_response_json(retrieved_2, "Retrieved Response (Instance 2)")
    log_result(
        "2a. Cross-Instance Retrieve — Instance 1 → Instance 2",
        success=True,
        details=f"Retrieved output: {retrieved_2.output_text}",
    )
except Exception as e:
    log_result(
        "2a. Cross-Instance Retrieve — Instance 1 → Instance 2",
        success=False,
        details=str(e),
    )

Attempting to retrieve response resp_0a1c9e7331341b450069cba80095588194bcc1c817f57699f5 from Instance 2...
[FAIL] 2a. Cross-Instance Retrieve — Instance 1 → Instance 2
       ↳ Error code: 404 - {'error': {'message': "Response with id 'resp_0a1c9e7331341b450069cba80095588194bcc1c817f57699f5' not found.", 'type': 'invalid_request_error', 'param': None, 'code': None}}


In [7]:
# Test 2b: Try to retrieve the response created on Instance 1 from Instance 3
print(f"Attempting to retrieve response {response1_id} from Instance 3...")
try:
    retrieved_3 = client3.responses.retrieve(response1_id)
    print_response_json(retrieved_3, "Retrieved Response (Instance 3)")
    log_result(
        "2b. Cross-Instance Retrieve — Instance 1 → Instance 3",
        success=True,
        details=f"Retrieved output: {retrieved_3.output_text}",
    )
except Exception as e:
    log_result(
        "2b. Cross-Instance Retrieve — Instance 1 → Instance 3",
        success=False,
        details=str(e),
    )

Attempting to retrieve response resp_0a1c9e7331341b450069cba80095588194bcc1c817f57699f5 from Instance 3...
[FAIL] 2b. Cross-Instance Retrieve — Instance 1 → Instance 3
       ↳ Error code: 404 - {'error': {'message': "Response with id 'resp_0a1c9e7331341b450069cba80095588194bcc1c817f57699f5' not found.", 'type': 'invalid_request_error', 'param': None, 'code': None}}


---
## 3. Cross-Instance Chaining — Use `previous_response_id` on Instance 2 & 3

In [8]:
# Test 3a: Chain on Instance 2 using response_id from Instance 1
print(f"Chaining on Instance 2 with previous_response_id={response1_id}...")
try:
    chained_2 = client2.responses.create(
        model=DEPLOYMENT,
        previous_response_id=response1_id,
        input=[{"role": "user", "content": "What is the population of that city? Answer briefly."}],
    )
    print_response_json(chained_2, "Chained Response (Instance 2)")
    log_result(
        "3a. Cross-Instance Chaining — Instance 1 → Instance 2",
        success=True,
        details=f"Chained output: {chained_2.output_text}",
    )
except Exception as e:
    log_result(
        "3a. Cross-Instance Chaining — Instance 1 → Instance 2",
        success=False,
        details=str(e),
    )

Chaining on Instance 2 with previous_response_id=resp_0a1c9e7331341b450069cba80095588194bcc1c817f57699f5...
[FAIL] 3a. Cross-Instance Chaining — Instance 1 → Instance 2
       ↳ Error code: 400 - {'error': {'message': "Previous response with id 'resp_0a1c9e7331341b450069cba80095588194bcc1c817f57699f5' not found.", 'type': 'invalid_request_error', 'param': 'previous_response_id', 'code': 'previous_response_not_found'}}


In [9]:
# Test 3b: Chain on Instance 3 using response_id from Instance 1
print(f"Chaining on Instance 3 with previous_response_id={response1_id}...")
try:
    chained_3 = client3.responses.create(
        model=DEPLOYMENT,
        previous_response_id=response1_id,
        input=[{"role": "user", "content": "What is the population of that city? Answer briefly."}],
    )
    print_response_json(chained_3, "Chained Response (Instance 3)")
    log_result(
        "3b. Cross-Instance Chaining — Instance 1 → Instance 3",
        success=True,
        details=f"Chained output: {chained_3.output_text}",
    )
except Exception as e:
    log_result(
        "3b. Cross-Instance Chaining — Instance 1 → Instance 3",
        success=False,
        details=str(e),
    )

Chaining on Instance 3 with previous_response_id=resp_0a1c9e7331341b450069cba80095588194bcc1c817f57699f5...
[FAIL] 3b. Cross-Instance Chaining — Instance 1 → Instance 3
       ↳ Error code: 400 - {'error': {'message': "Previous response with id 'resp_0a1c9e7331341b450069cba80095588194bcc1c817f57699f5' not found.", 'type': 'invalid_request_error', 'param': 'previous_response_id', 'code': 'previous_response_not_found'}}


---
## 4. Background Mode — Baseline (same instance)

In [10]:
# Test 4: Create a background response on Instance 1, poll until complete, retrieve from same instance
print("Creating background response on Instance 1...")
bg_response = client1.responses.create(
    model=DEPLOYMENT,
    input="Explain quantum entanglement in three sentences.",
    background=True,
)

bg_response_id = bg_response.id
print(f"Background Response ID : {bg_response_id}")
print(f"Initial Status         : {bg_response.status}")
print_response_json(bg_response, "Background Create Response (Instance 1)")

# Poll until completion
print("Polling for completion on Instance 1...")
try:
    completed_bg = wait_for_background(client1, bg_response_id)
    print_response_json(completed_bg, "Background Completed Response (Instance 1)")
    log_result(
        "4. Background Baseline — create & poll on Instance 1",
        success=True,
        details=f"Final status: {completed_bg.status} | Output: {completed_bg.output_text[:200]}",
    )
except Exception as e:
    log_result("4. Background Baseline — create & poll on Instance 1", success=False, details=str(e))

Creating background response on Instance 1...
Background Response ID : resp_03157dbc811859c70069cba842ba308195959c28d709659331
Initial Status         : queued

📋 Background Create Response (Instance 1) (JSON):
{
  "id": "resp_03157dbc811859c70069cba842ba308195959c28d709659331",
  "created_at": 1774954562.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4-1-mini",
  "object": "response",
  "output": [],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 1.0,
  "background": true,
  "completed_at": null,
  "conversation": null,
  "max_output_tokens": null,
  "max_tool_calls": null,
  "previous_response_id": null,
  "prompt": null,
  "prompt_cache_key": null,
  "prompt_cache_retention": null,
  "reasoning": {
    "effort": null,
    "generate_summary": null,
    "summary": null
  },
  "safety_identifier": null,
  "service_tier": "auto",
  "status": "queued",
  "text": {
    "for

In [ ]:
# Test 4b: Explicitly retrieve the completed background response from Instance 1 (same instance)
print(f"Retrieving background response {bg_response_id} from Instance 1 (same instance)...")
try:
    bg_retrieved_1 = client1.responses.retrieve(bg_response_id)
    print_response_json(bg_retrieved_1, "Background Retrieved (Instance 1 — same instance)")
    log_result(
        "4b. Background Retrieve — same Instance 1",
        success=True,
        details=f"Status: {bg_retrieved_1.status} | Output: {bg_retrieved_1.output_text[:200]}",
    )
except Exception as e:
    log_result(
        "4b. Background Retrieve — same Instance 1",
        success=False,
        details=str(e),
    )

---
## 5. Background Cross-Instance Retrieve

In [11]:
# Test 5a: Retrieve the background response (created on Instance 1) from Instance 2
print(f"Attempting to retrieve background response {bg_response_id} from Instance 2...")
try:
    bg_retrieved_2 = client2.responses.retrieve(bg_response_id)
    print_response_json(bg_retrieved_2, "Background Retrieved (Instance 2)")
    log_result(
        "5a. Background Cross-Instance Retrieve — Instance 1 → Instance 2",
        success=True,
        details=f"Status: {bg_retrieved_2.status} | Output: {bg_retrieved_2.output_text[:200]}",
    )
except Exception as e:
    log_result(
        "5a. Background Cross-Instance Retrieve — Instance 1 → Instance 2",
        success=False,
        details=str(e),
    )

Attempting to retrieve background response resp_03157dbc811859c70069cba842ba308195959c28d709659331 from Instance 2...
[FAIL] 5a. Background Cross-Instance Retrieve — Instance 1 → Instance 2
       ↳ Error code: 404 - {'error': {'message': "Response with id 'resp_03157dbc811859c70069cba842ba308195959c28d709659331' not found.", 'type': 'invalid_request_error', 'param': None, 'code': None}}


In [12]:
# Test 5b: Retrieve the background response (created on Instance 1) from Instance 3
print(f"Attempting to retrieve background response {bg_response_id} from Instance 3...")
try:
    bg_retrieved_3 = client3.responses.retrieve(bg_response_id)
    print_response_json(bg_retrieved_3, "Background Retrieved (Instance 3)")
    log_result(
        "5b. Background Cross-Instance Retrieve — Instance 1 → Instance 3",
        success=True,
        details=f"Status: {bg_retrieved_3.status} | Output: {bg_retrieved_3.output_text[:200]}",
    )
except Exception as e:
    log_result(
        "5b. Background Cross-Instance Retrieve — Instance 1 → Instance 3",
        success=False,
        details=str(e),
    )

Attempting to retrieve background response resp_03157dbc811859c70069cba842ba308195959c28d709659331 from Instance 3...
[FAIL] 5b. Background Cross-Instance Retrieve — Instance 1 → Instance 3
       ↳ Error code: 404 - {'error': {'message': "Response with id 'resp_03157dbc811859c70069cba842ba308195959c28d709659331' not found.", 'type': 'invalid_request_error', 'param': None, 'code': None}}


---
## 6. Background Cross-Instance Chaining

In [13]:
# Test 6a: Chain on Instance 2 using the background response_id from Instance 1
print(f"Chaining on Instance 2 with background previous_response_id={bg_response_id}...")
try:
    bg_chained_2 = client2.responses.create(
        model=DEPLOYMENT,
        previous_response_id=bg_response_id,
        input=[{"role": "user", "content": "Can you give a real-world analogy for that concept?"}],
    )
    print_response_json(bg_chained_2, "Background Chained Response (Instance 2)")
    log_result(
        "6a. Background Cross-Instance Chaining — Instance 1 → Instance 2",
        success=True,
        details=f"Chained output: {bg_chained_2.output_text[:200]}",
    )
except Exception as e:
    log_result(
        "6a. Background Cross-Instance Chaining — Instance 1 → Instance 2",
        success=False,
        details=str(e),
    )

Chaining on Instance 2 with background previous_response_id=resp_03157dbc811859c70069cba842ba308195959c28d709659331...
[FAIL] 6a. Background Cross-Instance Chaining — Instance 1 → Instance 2
       ↳ Error code: 400 - {'error': {'message': "Previous response with id 'resp_03157dbc811859c70069cba842ba308195959c28d709659331' not found.", 'type': 'invalid_request_error', 'param': 'previous_response_id', 'code': 'previous_response_not_found'}}


In [14]:
# Test 6b: Chain on Instance 3 using the background response_id from Instance 1
print(f"Chaining on Instance 3 with background previous_response_id={bg_response_id}...")
try:
    bg_chained_3 = client3.responses.create(
        model=DEPLOYMENT,
        previous_response_id=bg_response_id,
        input=[{"role": "user", "content": "Can you give a real-world analogy for that concept?"}],
    )
    print_response_json(bg_chained_3, "Background Chained Response (Instance 3)")
    log_result(
        "6b. Background Cross-Instance Chaining — Instance 1 → Instance 3",
        success=True,
        details=f"Chained output: {bg_chained_3.output_text[:200]}",
    )
except Exception as e:
    log_result(
        "6b. Background Cross-Instance Chaining — Instance 1 → Instance 3",
        success=False,
        details=str(e),
    )

Chaining on Instance 3 with background previous_response_id=resp_03157dbc811859c70069cba842ba308195959c28d709659331...
[FAIL] 6b. Background Cross-Instance Chaining — Instance 1 → Instance 3
       ↳ Error code: 400 - {'error': {'message': "Previous response with id 'resp_03157dbc811859c70069cba842ba308195959c28d709659331' not found.", 'type': 'invalid_request_error', 'param': 'previous_response_id', 'code': 'previous_response_not_found'}}


---
## 7. Summary of Results

In [15]:
print("=" * 90)
print("RESPONSE API STATELESS CROSS-INSTANCE TEST RESULTS")
print("=" * 90)
print(f"{'Test':<62} {'Status':<8} Details")
print("-" * 90)
for r in results:
    colour = "\033[92m" if r['status'] == 'PASS' else "\033[91m"
    # Truncate details for the summary table
    short_details = (r['details'][:60] + '...') if len(r['details']) > 60 else r['details']
    print(f"{r['test']:<62} {colour}{r['status']:<8}\033[0m {short_details}")
print("-" * 90)

passed = sum(1 for r in results if r['status'] == 'PASS')
failed = sum(1 for r in results if r['status'] == 'FAIL')
print(f"\nTotal: {len(results)} | Passed: {passed} | Failed: {failed}")

if failed > 0:
    print("\n⚠️  Some tests FAILED — response state is NOT shared across Foundry instances.")
    print("   This confirms that each Azure OpenAI resource maintains its own response storage.")
else:
    print("\n✅ All tests PASSED — response state IS shared across Foundry instances.")

RESPONSE API STATELESS CROSS-INSTANCE TEST RESULTS
Test                                                           Status   Details
------------------------------------------------------------------------------------------
1. Baseline — create & retrieve on Instance 1                  PASS     Retrieved output: Paris
2a. Cross-Instance Retrieve — Instance 1 → Instance 2          FAIL     Error code: 404 - {'error': {'message': "Response with id 'r...
2b. Cross-Instance Retrieve — Instance 1 → Instance 3          FAIL     Error code: 404 - {'error': {'message': "Response with id 'r...
3a. Cross-Instance Chaining — Instance 1 → Instance 2          FAIL     Error code: 400 - {'error': {'message': "Previous response w...
3b. Cross-Instance Chaining — Instance 1 → Instance 3          FAIL     Error code: 400 - {'error': {'message': "Previous response w...
4. Background Baseline — create & poll on Instance 1           PASS     Final status: completed | Output: Quantum entanglement is a ...
5a

---
## 8. Cleanup (Optional)

Delete responses created during testing to keep storage clean.

In [ ]:
# Cleanup: Delete responses from Instance 1
print("Cleaning up responses on Instance 1...")
try:
    client1.responses.delete(response1_id)
    print(f"  Deleted {response1_id}")
except Exception as e:
    print(f"  Could not delete {response1_id}: {e}")

try:
    client1.responses.delete(bg_response_id)
    print(f"  Deleted {bg_response_id}")
except Exception as e:
    print(f"  Could not delete {bg_response_id}: {e}")

print("Done.")